In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
%cd /content/drive/MyDrive/ColabNotebooks/capstone/Final_Report

/content/drive/MyDrive/ColabNotebooks/capstone/Final_Report


In [16]:
import pandas as pd
import numpy as np
import pickle
import gradio as gr
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
!pip install gradio

In [18]:
m_df = pd.read_csv('processed_movies.csv')

m_feats = np.load('movies_features.npy')

with open('final_recommendation_model.pkl', 'rb') as f:
    model_config = pickle.load(f)
    id_map = model_config['id_map']

print("Demo Data Loaded Successfully!")

Demo Data Loaded Successfully!


In [ ]:
import gradio as gr
from openai import OpenAI
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import math

# --- 1. Basic Settings ---
# Note: Ensure your GITHUB_TOKEN is kept secure.
GITHUB_TOKEN = "need_api_key"
client = OpenAI(base_url="https://models.inference.ai.azure.com", api_key=GITHUB_TOKEN)

initial_profile = {"liked_titles": ["Toy Story", "GoldenEye"]}

# --- 2. Indicator Calculation Assistance ---
def calculate_metrics(rec_list, profile_likes):
    """Calculates the overlap between recommendation results and User Profile"""
    if not rec_list: return 0.0, 0.0
    # Relevance Definition: Whether the recommended movie is in the user's liked list.
    rel_list = [1 if t in profile_likes else 0 for t in rec_list]

    p = sum(rel_list) / len(rec_list)
    dcg = sum([rel / math.log2(i + 2) for i, rel in enumerate(rel_list)])
    ideal_rel = sorted(rel_list, reverse=True)
    idcg = sum([rel / math.log2(i + 2) for i, rel in enumerate(ideal_rel)])
    n = dcg / idcg if idcg > 0 else 0
    return p, n

# --- 3. Recommendation Engine ---
def run_recommendation(seeds, alpha=0.7, k=10, target_genre=None, offset=0):
    try:
        # Assuming the first 5000 features are TF-IDF and the rest are Genre One-Hot
        tfidf_part = m_feats[:, :5000]
        genre_part = m_feats[:, 5000:]

        seed_indices = [m_df[m_df['title'] == t].index[0] for t in seeds if t in m_df['title'].values]
        if not seed_indices: return []

        c_sim = np.mean([cosine_similarity(tfidf_part[i].reshape(1, -1), tfidf_part).flatten() for i in seed_indices], axis=0)
        g_sim = np.mean([cosine_similarity(genre_part[i].reshape(1, -1), genre_part).flatten() for i in seed_indices], axis=0)
        final_scores = (alpha * c_sim) + ((1 - alpha) * g_sim)

        if target_genre:
            mask = m_df['genres'].str.contains(target_genre, case=False, na=False)
            final_scores[mask] += 0.3

        # Sort and select, supporting offset to fetch different batches
        rec_indices = final_scores.argsort()[-(k+len(seeds)+offset):-(1+offset)][::-1]
        rec_titles = [m_df.iloc[i]['title'] for i in rec_indices if m_df.iloc[i]['title'] not in seeds][:k]
        return rec_titles
    except:
        return []

# --- 4. Core Dialogue Processing Logic ---
def handle_interaction(message, history, profile, alpha, pending_movie, last_recs, mode):
    # Initialize default return values (7 items to match UI outputs)
    res_msg, res_hist, res_prof, res_table, res_pend, res_l_recs, res_mode = "", history, profile, None, None, last_recs, None

    # 【Scenario A: Handling the "Re-recommendation" mode selection】
    if mode == "awaiting_rec_choice":
        if "AI" in message.upper():
            prompt = f"User profile likes: {profile['liked_titles']}. The model just recommended these 10: {last_recs}. Please select 5 and explain why."
            ai_res = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}], model="gpt-4o-mini"
            ).choices[0].message.content

            llm_rec_5 = last_recs[:5]
            p, n = calculate_metrics(llm_rec_5, profile["liked_titles"])

            res_hist.append({"role": "user", "content": message})
            res_hist.append({"role": "assistant", "content": f"【LLM AI Selection Mode】\n{ai_res}"})
            res_table = [["Precision", "0.0320", "0.0360", f"{p:.4f} (AI)"], ["nDCG", "0.5115", "0.5378", f"{n:.4f} (AI)"]]
            return res_msg, res_hist, res_prof, res_table, None, llm_rec_5, None

        elif "model" in message.lower() or "recommender" in message.lower():
            new_recs = run_recommendation(profile["liked_titles"], alpha=alpha, offset=10)
            p, n = calculate_metrics(new_recs, profile["liked_titles"])
            res_text = "【Recommender Model】New batch of recommendations for you:\n" + "\n".join([f"- {t}" for t in new_recs])
            res_hist.append({"role": "user", "content": message})
            res_hist.append({"role": "assistant", "content": res_text})
            res_table = [["Precision", "0.0320", "0.0360", f"{p:.4f}"], ["nDCG", "0.5115", "0.5378", f"{n:.4f}"]]
            return res_msg, res_hist, res_prof, res_table, None, new_recs, None

    # [Scenario B: Handling inquiries about adding a single movie to a profile]
    if pending_movie:
        if any(word in message.lower() for word in ["yes", "yeah", "sure", "correct"]):
            res_prof["liked_titles"].append(pending_movie)
            recs = run_recommendation(res_prof["liked_titles"], alpha=alpha)

            prompt = f"User just added '{pending_movie}' to Profile. Current Profile: {res_prof['liked_titles']}. Recs: {recs[:3]}. Explain how these match their long-term preferences."
            ai_res = client.chat.completions.create(messages=[{"role":"user", "content":prompt}], model="gpt-4o-mini").choices[0].message.content
            res_text = f"✅ Profile updated!\n\n{ai_res}"
        else:
            recs = run_recommendation([pending_movie], alpha=alpha)
            prompt = f"Please disregard history and recommend these films solely based on the style of {pending_movie}: {recs[:3]}. Please act like a neutral film critic."
            ai_res = client.chat.completions.create(messages=[{"role":"user", "content":prompt}], model="gpt-4o-mini").choices[0].message.content
            res_text = f"💡 We'll delve deeper into the details of {pending_movie}:\n\n{ai_res}"

        p, n = calculate_metrics(recs, res_prof["liked_titles"])
        res_hist.append({"role": "user", "content": message})
        res_hist.append({"role": "assistant", "content": res_text})
        res_table = [["Precision", "0.0320", "0.0360", f"{p:.4f}"], ["nDCG", "0.5115", "0.5378", f"{n:.4f}"]]
        return res_msg, res_hist, res_prof, res_table, None, recs, None

    # 【Situation C: Triggering "Re-recommendation"】
    if "recommend" in message.lower() and ("again" in message.lower() or "re-" in message.lower()):
        res_text = "Would you like to use **AI Selection** (pick 5 best from the previous results) or **Recommender Model** (fetch a completely new batch of movies)?"
        res_hist.append({"role": "user", "content": message})
        res_hist.append({"role": "assistant", "content": res_text})
        return res_msg, res_hist, res_prof, None, None, last_recs, "awaiting_rec_choice"

    # 【Scenario D: Detect whether the input is a known movie title】
    if message in m_df['title'].values:
        res_text = f"Detected movie: '{message}'.\nWould you like to 【Add to Profile】 to optimize long-term results, or 【Recommend based only on this movie】? (Yes/No)"
        res_hist.append({"role": "user", "content": message})
        res_hist.append({"role": "assistant", "content": res_text})
        return res_msg, res_hist, res_prof, None, message, [], None

    # 【Scenario E: General Needs/Genre Search】
    genre_keyword = message.lower().replace("recommend", "").replace("movie", "").replace("some", "").strip()
    recs = run_recommendation(profile["liked_titles"], alpha=alpha, target_genre=genre_keyword)
    p, n = calculate_metrics(recs, profile["liked_titles"])

    prompt = f"Request: {message}. Profile: {profile['liked_titles']}. Recommendations: {recs[:3]}. Provide reasons."
    ai_res = client.chat.completions.create(messages=[{"role":"user", "content":prompt}], model="gpt-4o-mini").choices[0].message.content

    res_hist.append({"role": "user", "content": message})
    res_hist.append({"role": "assistant", "content": ai_res})
    res_table = [["Precision", "0.0320", "0.0360", f"{p:.4f}"], ["nDCG", "0.5115", "0.5378", f"{n:.4f}"]]
    return res_msg, res_hist, res_prof, res_table, None, recs, None

# --- 5. UI Interface ---
custom_css = """
.gradio-container { background-color: #ffffff !important; }
.black_btn { border: 2px solid black !important; border-radius: 0px !important; font-weight: bold !important; background: white !important; }
.black_btn:hover { background: black !important; color: white !important; }
"""

with gr.Blocks(css=custom_css) as demo:
    state_profile = gr.State(initial_profile)
    state_pending = gr.State(None)
    state_last_recs = gr.State([])
    state_mode = gr.State(None)

    gr.Markdown("# Intelligent Movie Dialogue Recommendation System")

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="AI Assistant", type="messages", height=500)
            msg = gr.Textbox(label="Enter movie title, genre, or commands (Yes/No/Recommend Again)")
            alpha_slider = gr.Slider(0, 1, 0.7, label="Weighting Balance (Plot <-> Genre)")
            btn = gr.Button("Send Command", elem_classes="black_btn")

        with gr.Column(scale=1):
            gr.Markdown("### Performance Metrics")
            metrics_table = gr.DataFrame(
                headers=["Indicators", "Benchmarks", "Optimization", "Real-time Evaluation"],
                value=[["Precision", "0.0320", "0.0360", "-"], ["nDCG", "0.5115", "0.5378", "-"]]
            )
            gr.Markdown("### 👤 User Profile (JSON)")
            profile_json = gr.JSON(value=initial_profile)

    # Event binding
    input_list = [msg, chatbot, state_profile, alpha_slider, state_pending, state_last_recs, state_mode]
    output_list = [msg, chatbot, state_profile, metrics_table, state_pending, state_last_recs, state_mode]

    btn.click(handle_interaction, input_list, output_list)
    msg.submit(handle_interaction, input_list, output_list)

    state_profile.change(lambda x: x, state_profile, profile_json)

demo.launch(debug=True)

/tmp/ipykernel_4591/518632691.py:139: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:
/tmp/ipykernel_4591/518632691.py:149: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="AI Assistant", type="messages", height=500)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://64919b626b70ec1300.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://64919b626b70ec1300.gradio.live
